In [ ]:
# Dash setup
from dash import Dash, dcc, html
import dash_leaflet as dl
from dash import dash_table
from dash.dependencies import Input, Output
import plotly.express as px

import pandas as pd
import base64

# Import enhanced CRUD
from ENHANCED_CRUD_Python_Module import AnimalShelter


# Load data
db = AnimalShelter()
records = db.read({})
df = pd.DataFrame(records)

# Empty data warning
if df.empty:
    print("Warning: No data loaded from MongoDB or CSV")
    df = pd.DataFrame(columns=[
        "animal_type", "breed", "name",
        "location_lat", "location_long"
    ])

# Clean column names safely
df.columns = df.columns.astype(str).str.strip()

# Convert coordinates safely
for col in ["location_lat", "location_long"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")


# App setup
app = Dash(__name__)

# Logo
image_filename = "Grazioso Salvare Logo.png"
encoded_image = base64.b64encode(open(image_filename, "rb").read())


# Layout
app.layout = html.Div([

    html.Center([
        html.Img(
            src="data:image/png;base64,{}".format(encoded_image.decode()),
            style={"width": "250px"}
        ),
        html.H1("CS-340 Dashboard"),
        html.H2("Grazioso Salvare Rescue Dashboard")
    ]),

    html.Hr(),

    dcc.RadioItems(
        id="filter-type",
        options=[
            {"label": "Water Rescue", "value": "WR"},
            {"label": "Mountain Rescue", "value": "MWR"},
            {"label": "Disaster Tracking", "value": "DIT"},
            {"label": "RESET", "value": "RESET"}
        ],
        value="RESET",
        inline=True
    ),

    html.Hr(),

    dash_table.DataTable(
        id="datatable-id",
        columns=[{"name": i, "id": i} for i in df.columns],
        data=df.to_dict("records"),
        page_size=10,
        sort_action="native",
        filter_action="native",
        row_selectable="single",
        selected_rows=[0]
    ),

    html.Br(),
    html.Hr(),

    html.Div([
        html.Div(id="graph-id", style={"width": "50%", "display": "inline-block"}),
        html.Div(id="map-id", style={"width": "50%", "display": "inline-block"})
    ])
])


# Filter callback
@app.callback(
    Output("datatable-id", "data"),
    Input("filter-type", "value")
)
def update_table(filter_type):

    if filter_type == "WR":
        query = {"animal_type": "Dog", "sex_upon_outcome": "Intact Female"}

    elif filter_type == "MWR":
        query = {"animal_type": "Dog", "sex_upon_outcome": "Intact Male"}

    elif filter_type == "DIT":
        query = {"animal_type": "Dog"}

    else:
        query = {}

    return db.read(query)


# Pie chart
@app.callback(
    Output("graph-id", "children"),
    Input("datatable-id", "data")
)
def update_graph(data):

    if not data:
        return []

    dff = pd.DataFrame(data)

    if "breed" not in dff.columns:
        return []

    fig = px.pie(dff, names="breed", title="Breed Distribution")

    return [dcc.Graph(figure=fig)]


# Map
@app.callback(
    Output("map-id", "children"),
    Input("datatable-id", "data"),
    Input("datatable-id", "selected_rows")
)
def update_map(data, selected_rows):

    if not data:
        return []

    dff = pd.DataFrame(data)

    row = selected_rows[0] if selected_rows else 0

    if row >= len(dff):
        row = 0

    if "location_lat" not in dff.columns or "location_long" not in dff.columns:
        return []

    return [
        dl.Map(
            style={"width": "100%", "height": "500px"},
            center=[30.75, -97.48],
            zoom=10,
            children=[
                dl.TileLayer(),

                dl.Marker(
                    position=[
                        dff.iloc[row]["location_lat"],
                        dff.iloc[row]["location_long"]
                    ],
                    children=[
                        dl.Tooltip(dff.iloc[row].get("breed", "Unknown")),
                        dl.Popup(html.H3(dff.iloc[row].get("name", "No Name")))
                    ]
                )
            ]
        )
    ]


# Run app
app.run(debug=True)

MongoDB connection failed, switching to CSV fallback: localhost:27017: [Errno 61] Connection refused (configured timeouts: socketTimeoutMS: 20000.0ms, connectTimeoutMS: 20000.0ms), Timeout: 3.0s, Topology Description: <TopologyDescription id: 6a25f2c509b681b4a97dac01, topology_type: Unknown, servers: [<ServerDescription ('localhost', 27017) server_type: Unknown, rtt: None, error=AutoReconnect('localhost:27017: [Errno 61] Connection refused (configured timeouts: socketTimeoutMS: 20000.0ms, connectTimeoutMS: 20000.0ms)')>]>
